1. Imports

In [1]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering as AggCls
from datetime import datetime
import numpy as np
from collections import Counter
import pandas as pd
import csv
from sklearn.cluster import KMeans
from transformers import BertTokenizer, BertModel
import torch
from sentence_transformers import SentenceTransformer, util

c:\Users\korey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bertErrorStories = []
bertCorrectStories = []

with open('./bertKmeansClusteringDistances.csv', mode='r', newline='', encoding='utf-8') as csv_file:
    
    thresholdAssign = 0

    csvReader = csv.reader(csv_file, delimiter=',') 

    withinTresholdCount = 0
    correctCount = 0
    incorrectCount = 0
    i = 0

    for row in csvReader:
        if i != 0:
            
            distanceMetricOne = float(row[2])
            distanceMetricTwo = float(row[4])
            correctEnding = int(row[5])

            if correctEnding == 1:
                if distanceMetricOne < distanceMetricTwo:
                    correctCount += 1
                    bertCorrectStories.append(row[0])
                else:
                    incorrectCount += 1
                    bertErrorStories.append(row[0])

            elif correctEnding == 2:
                if distanceMetricTwo < distanceMetricOne:
                    correctCount += 1
                    bertCorrectStories.append(row[0])
                else:
                    incorrectCount += 1
                    bertErrorStories.append(row[0])

        i = i + 1

In [3]:
print("BERT Model\nTotal Stories Processed: {}\nCount of Stories Classified Accurately: {}\nCount of Stories Classified Inaccurately: {}\nOverall Accuracy: {}%".format(correctCount + incorrectCount, correctCount, incorrectCount, (correctCount / (correctCount + incorrectCount) * 100)))

BERT Model
Total Stories Processed: 1571
Count of Stories Classified Accurately: 762
Count of Stories Classified Inaccurately: 809
Overall Accuracy: 48.504137492043284%


In [4]:
sbertErrorStories = []
sbertCorrectStories = []

with open('./sbertKmeansClusteringDistances.csv', mode='r', newline='', encoding='utf-8') as csv_file:
    
    thresholdAssign = 0

    csvReader = csv.reader(csv_file, delimiter=',') 

    #withinTresholdCount = 0
    correctCount = 0
    incorrectCount = 0
    i = 0

    for row in csvReader:
        if i != 0:
            
            distanceMetricOne = float(row[3])
            distanceMetricTwo = float(row[5])
            correctEnding = int(row[6])

          
            if correctEnding == 1:
                if distanceMetricOne < distanceMetricTwo:
                    correctCount += 1
                    sbertCorrectStories.append(row[0])
                else:
                    incorrectCount += 1
                    sbertErrorStories.append(row[0])

            elif correctEnding == 2:
                if distanceMetricTwo < distanceMetricOne:
                    correctCount += 1
                    sbertCorrectStories.append(row[0])
                else:
                    incorrectCount += 1
                    sbertErrorStories.append(row[0])

        i = i + 1

In [5]:
print("S-BERT Model\nTotal Stories Processed: {}\nCount of Stories Classified Accurately: {}\nCount of Stories Classified Inaccurately: {}\nOverall Accuracy: {}%".format(correctCount + incorrectCount, correctCount, incorrectCount, (correctCount / (correctCount + incorrectCount) * 100)))

S-BERT Model
Total Stories Processed: 1571
Count of Stories Classified Accurately: 964
Count of Stories Classified Inaccurately: 607
Overall Accuracy: 61.362189688096755%


In [6]:
CLOZE_DF = pd.read_csv("D://Python//MAForumProject//cloze_winter_2018_val.csv", sep = ",")
correctAnsList = list(CLOZE_DF['AnswerRightEnding'])
print(len(correctAnsList))

1571


In [7]:
s3bertCorrectStories = []
s3bertErrorStories = []

with open('./s3bertKmeansClusteringDistances.csv') as csv_file:
    
    
    csv_reader = csv.reader(csv_file, delimiter=',') 

    correct_count = 0
    incorrect_count = 0

    i = 0
    for row in csv_reader:
        if i:
            #print(i)
            if correctAnsList[i-1] == 1: #Note this value is subtracted by 1 since the correct_ans_list is a separate array declared elsewhere. The (i-1) factor accounts for the excelheader.
                if float(row[1]) < float(row[3]):
                    correct_count += 1
                    
                    for j, rows in CLOZE_DF.iterrows():
                        if str(row[0]) == str(rows['RandomFifthSentenceQuiz1']):
                            s3bertCorrectStories.append(rows['InputStoryid'])
                            #print(1)
                            break

                elif float(row[3]) < float(row[1]):
                    incorrect_count += 1

                    for j, rows in CLOZE_DF.iterrows():
                        if str(row[0]) == str(rows['RandomFifthSentenceQuiz1']):
                            s3bertErrorStories.append(rows['InputStoryid'])
                            #print(2)
                            break

            elif correctAnsList[i-1] == 2:

                if float(row[1]) < float(row[3]):
                    incorrect_count += 1

                    for j, rows in CLOZE_DF.iterrows():
                        if str(row[0]) == str(rows['RandomFifthSentenceQuiz1']):
                            s3bertErrorStories.append(rows['InputStoryid'])
                            #print(3)
                            break

                elif float(row[3]) < float(row[1]):
                    correct_count += 1

                    for j, rows in CLOZE_DF.iterrows():
                        if str(row[0]) == str(rows['RandomFifthSentenceQuiz1']):
                            s3bertCorrectStories.append(rows['InputStoryid'])
                            #print(4)
                            break
        
        i = i + 1

        if i % 100 == 0:
            print("{} Stories Processed With S3BERT".format(i))
    print("{} Stories Processed with S3BERT".format(i - 1))

100 Stories Processed With S3BERT
200 Stories Processed With S3BERT
300 Stories Processed With S3BERT
400 Stories Processed With S3BERT
500 Stories Processed With S3BERT
600 Stories Processed With S3BERT
700 Stories Processed With S3BERT
800 Stories Processed With S3BERT
900 Stories Processed With S3BERT
1000 Stories Processed With S3BERT
1100 Stories Processed With S3BERT
1200 Stories Processed With S3BERT
1300 Stories Processed With S3BERT
1400 Stories Processed With S3BERT
1500 Stories Processed With S3BERT
1571 Stories Processed with S3BERT


In [8]:
print("S3BERT Model\nTotal Stories Processed: {}\nCount of Stories Classified Accurately: {}\nCount of Stories Classified Inaccurately: {}\nOverall Accuracy: {}%".format(correct_count + incorrect_count, correct_count, incorrect_count, (correct_count / (correct_count + incorrect_count) * 100)))

S3BERT Model
Total Stories Processed: 1571
Count of Stories Classified Accurately: 1019
Count of Stories Classified Inaccurately: 552
Overall Accuracy: 64.8631444939529%


2. Error Analysis

2.1. Clustering Error Analysis

In [9]:
allThreeErrorList = []

bertS3bertErrorList = []

bertSbertErrorList = []

sbertS3bertErrorList = []

onlyS3bertErrorList = []

onlySBertErrorList = []

onlyBertErrorList= []


i = 0

for row in CLOZE_DF.iterrows():
    #print(row[1][0])
    #print(bertErrorStories)
    #print(row[1])
    if row[1]['InputStoryid'] in bertErrorStories and row[1]['InputStoryid'] in sbertErrorStories and row[1]['InputStoryid'] in s3bertErrorStories: #BERT, SBERT, and S3BERT Errors
        allThreeErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertErrorStories and row[1]['InputStoryid'] in sbertCorrectStories and row[1]['InputStoryid'] in s3bertErrorStories: #BERT & S3BERT Errors
        bertS3bertErrorList.append(row[1]['InputStoryid'])
        i+= 1
    elif row[1]['InputStoryid'] in bertCorrectStories and row[1]['InputStoryid'] in sbertErrorStories and row[1]['InputStoryid'] in s3bertErrorStories: #SBERT & S3BERT Errors
        sbertS3bertErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertErrorStories and row[1]['InputStoryid'] in sbertErrorStories and row[1]['InputStoryid'] in s3bertCorrectStories:
        bertSbertErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertCorrectStories and row[1]['InputStoryid'] in sbertCorrectStories and row[1]['InputStoryid'] in s3bertErrorStories: #S3BERT Errors
        onlyS3bertErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertErrorStories and row[1]['InputStoryid'] in sbertCorrectStories and row[1]['InputStoryid'] in s3bertCorrectStories: #BERT Errors
        onlyBertErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertCorrectStories and row[1]['InputStoryid'] in sbertErrorStories and row[1]['InputStoryid'] in s3bertCorrectStories: #SBERT Errors
        onlySBertErrorList.append(row[1]['InputStoryid'])
        i += 1
print(i)
print("All 3 Errors: {}\nBERT & S3BERT Errors: {}\nSBERT & S3BERT Errors: {}\nBERT & SBERT Errors: {}\nS3BERT Only Errors: {}\nBERT Only Errors: {}\nSBERT Only Errors: {}\nTotal Count of Error Stories: {}".format(len(allThreeErrorList), len(bertS3bertErrorList), len(sbertS3bertErrorList), len(bertSbertErrorList), len(onlyS3bertErrorList), len(onlyBertErrorList), len(onlySBertErrorList), len(allThreeErrorList) + len(bertS3bertErrorList) + len(sbertS3bertErrorList) + len(bertSbertErrorList) + len(onlyS3bertErrorList) + len(onlyBertErrorList) + len(onlySBertErrorList)))


1079
All 3 Errors: 302
BERT & S3BERT Errors: 31
SBERT & S3BERT Errors: 185
BERT & SBERT Errors: 69
S3BERT Only Errors: 34
BERT Only Errors: 407
SBERT Only Errors: 51
Total Count of Error Stories: 1079


2.2. Similarity Error Analysis

2.2.1. BERT Stories

In [10]:
bertEndingsDf = pd.read_csv('./bertSentBySentCosSims.csv')
bertEndingsDf.head(5)

,StoryID,First Ending Sent,S4E1CosSims,Second Ending Sent,S4E2CosSims,Correct Answer
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,He is happy now.,0.522551,He joined a gang.,0.472550,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,The brownies are so delicious Laverne eats two...,0.685989,Laverne doesn't go to her friend's party.,0.693997,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,Sarah then decided to move to Europe.,0.530687,Sarah decided that she preferred her home over...,0.691620,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina liked the cookies so much she ate them al...,0.557086,Gina gave the cookies away at her church.,0.507014,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,I was very proud of my performance.,0.820335,I was very ashamed of my performance.,0.619275,1


In [11]:
bertSentbySentCorrectStories = []
bertSentbySentErrorStories = []

In [12]:
correctCount = 0
incorrectCount = 0
i = 0
for row in bertEndingsDf.iterrows():
        #print(row)
        # Skip empty rows or malformed lines
            
        correct_answer = int(row[1]['Correct Answer'])

        if correct_answer == 1:
            if float(row[1]['S4E1CosSims']) > float(row[1]['S4E2CosSims']):
                correctCount += 1
                bertSentbySentCorrectStories.append(row[1]['StoryID'])
            elif float(row[1]['S4E2CosSims']) > float(row[1]['S4E1CosSims']):
                incorrectCount += 1
                bertSentbySentErrorStories.append(row[1]['StoryID'])
        elif correct_answer == 2:
            if float(row[1]['S4E1CosSims']) > float(row[1]['S4E2CosSims']):
                incorrectCount += 1
                bertSentbySentErrorStories.append(row[1]['StoryID'])
            elif float(row[1]['S4E2CosSims']) > float(row[1]['S4E1CosSims']):
                correctCount += 1
                bertSentbySentCorrectStories.append(row[1]['StoryID'])

        i = i + 1

In [13]:
print(correctCount, incorrectCount, incorrectCount + correctCount)


778 793 1571


2.2.2. S-BERT Stories

In [14]:
sbertEndingsDf = pd.read_csv("./sbertSentBySentCosSims.csv")
sbertEndingsDf.head(5)

,storyID,S4E1 Similarities,S4E2 Similarities,Correct Ending
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,0.296945,0.336776,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,0.842814,0.444635,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,0.272204,0.368684,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,0.833238,0.709119,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,0.573658,0.486423,1


In [16]:
correctCount = 0
incorrectCount = 0

sbertSentbySentCorrectStories = []
sbertSentbySentErrorStories = []

clozeStoryIDs = list(CLOZE_DF["InputStoryid"])


for row in sbertEndingsDf.iterrows():
        #print(row)
        # Skip empty rows or malformed lines
            
    correct_answer = int(row[1]['Correct Ending'])

    if correct_answer == 1:
        if float(row[1]['S4E1 Similarities']) > float(row[1]['S4E2 Similarities']):
            correctCount += 1
            sbertSentbySentCorrectStories.append(row[1]['storyID'])
        elif float(row[1]['S4E2 Similarities']) > float(row[1]['S4E1 Similarities']):
            incorrectCount += 1
            sbertSentbySentErrorStories.append(row[1]['storyID'])
    elif correct_answer == 2:
        if float(row[1]['S4E1 Similarities']) > float(row[1]['S4E2 Similarities']):
            incorrectCount += 1
            sbertSentbySentErrorStories.append(row[1]['storyID'])
        elif float(row[1]['S4E2 Similarities']) > float(row[1]['S4E1 Similarities']):
            correctCount += 1
            sbertSentbySentCorrectStories.append(row[1]['storyID'])

    i = i + 1

In [17]:
print(correctCount, incorrectCount, incorrectCount + correctCount)
print(len(sbertSentbySentCorrectStories), len(sbertSentbySentErrorStories))

980 591 1571
980 591


2.2.4. S3BERT Stories

In [18]:
s3bertEndingsDf = pd.read_csv("./s3bertSentBySentCosSims.csv")
s3bertEndingsDf['Correct Ending'] = list(sbertEndingsDf['Correct Ending'])
s3bertEndingsDf.head(5)

,storyID,CosSimsS4E1,CosSimsS4E2,Correct Ending
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,0.297691,0.390822,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,0.770141,0.398180,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,0.277267,0.401914,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,0.718231,0.585659,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,0.590311,0.532475,1


In [20]:
correctCount = 0
incorrectCount = 0

s3bertSentbySentCorrectStories = []
s3bertSentbySentErrorStories = []

clozeStoryIDs = list(CLOZE_DF["InputStoryid"])


for row in s3bertEndingsDf.iterrows():
            
    correct_answer = int(row[1]['Correct Ending'])

    if correct_answer == 1:
        if float(row[1]['CosSimsS4E1']) > float(row[1]['CosSimsS4E2']):
            correctCount += 1
            s3bertSentbySentCorrectStories.append(row[1]['storyID'])
        elif float(row[1]['CosSimsS4E2']) > float(row[1]['CosSimsS4E1']):
            incorrectCount += 1
            s3bertSentbySentErrorStories.append(row[1]['storyID'])
    elif correct_answer == 2:
        if float(row[1]['CosSimsS4E1']) > float(row[1]['CosSimsS4E2']):
            incorrectCount += 1
            s3bertSentbySentErrorStories.append(row[1]['storyID'])
        elif float(row[1]['CosSimsS4E2']) > float(row[1]['CosSimsS4E1']):
            correctCount += 1
            s3bertSentbySentCorrectStories.append(row[1]['storyID'])

    i = i + 1

In [21]:
print(correctCount, incorrectCount, incorrectCount + correctCount)
print(len(s3bertSentbySentCorrectStories), len(s3bertSentbySentErrorStories))

1013 558 1571
1013 558


2.2.5. Similarity Errors

In [23]:
allThreeSentbySentErrorList = []

bertS3bertSentbySentErrorList = []

bertSbertSentbySentErrorList = []

sbertS3bertSentbySentErrorList = []

onlyS3bertSentbySentErrorList = []

onlySBertSentbySentErrorList = []

onlyBertSentbySentErrorList= []


i = 0

for row in CLOZE_DF.iterrows():
    #print(row[1][0])
    #print(bertErrorStories)
    if row[1]['InputStoryid'] in bertSentbySentErrorStories and row[1]['InputStoryid'] in sbertSentbySentErrorStories and row[1]['InputStoryid'] in s3bertSentbySentErrorStories: #BERT, SBERT, and S3BERT Errors
        allThreeSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertSentbySentErrorStories and row[1]['InputStoryid'] in sbertSentbySentCorrectStories and row[1]['InputStoryid'] in s3bertSentbySentErrorStories: #BERT & S3BERT Errors
        bertS3bertSentbySentErrorList.append(row[1]['InputStoryid'])
        i+= 1
    elif row[1]['InputStoryid'] in bertSentbySentCorrectStories and row[1]['InputStoryid'] in sbertSentbySentErrorStories and row[1]['InputStoryid'] in s3bertSentbySentErrorStories: #SBERT & S3BERT Errors
        sbertS3bertSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertSentbySentErrorStories and row[1]['InputStoryid'] in sbertSentbySentErrorStories and row[1]['InputStoryid'] in s3bertSentbySentCorrectStories:
        bertSbertSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertSentbySentCorrectStories and row[1]['InputStoryid'] in sbertSentbySentCorrectStories and row[1]['InputStoryid'] in s3bertSentbySentErrorStories: #S3BERT Errors
        onlyS3bertSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertSentbySentErrorStories and row[1]['InputStoryid'] in sbertSentbySentCorrectStories and row[1]['InputStoryid'] in s3bertSentbySentCorrectStories: #BERT Errors
        onlyBertSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
    elif row[1]['InputStoryid'] in bertSentbySentCorrectStories and row[1]['InputStoryid'] in sbertSentbySentErrorStories and row[1]['InputStoryid'] in s3bertSentbySentCorrectStories: #SBERT Errors
        onlySBertSentbySentErrorList.append(row[1]['InputStoryid'])
        i += 1
#print(i)
print("All 3 Errors: {}\nBERT & S3BERT Errors: {}\nSBERT & S3BERT Errors: {}\nBERT & SBERT Errors: {}\nS3BERT Only Errors: {}\nBERT Only Errors: {}\nSBERT Only Errors: {}\nTotal Count of Error Stories: {}".format(len(allThreeSentbySentErrorList), len(bertS3bertSentbySentErrorList), len(sbertS3bertSentbySentErrorList), len(bertSbertSentbySentErrorList), len(onlyS3bertSentbySentErrorList), len(onlyBertSentbySentErrorList), len(onlySBertSentbySentErrorList), len(allThreeSentbySentErrorList) + len(bertS3bertSentbySentErrorList) + len(sbertS3bertSentbySentErrorList) + len(bertSbertSentbySentErrorList) + len(onlyS3bertSentbySentErrorList) + len(onlyBertSentbySentErrorList) + len(onlySBertSentbySentErrorList)))

All 3 Errors: 294
BERT & S3BERT Errors: 37
SBERT & S3BERT Errors: 175
BERT & SBERT Errors: 76
S3BERT Only Errors: 52
BERT Only Errors: 386
SBERT Only Errors: 46
Total Count of Error Stories: 1066


3. Overlapping Error Stories Between Metrics

3.1. Both Metrics Overlapping S-BERT Unique

In [24]:
sbertOverlap = list(set(onlySBertErrorList) & set(onlySBertSentbySentErrorList))

print(len(sbertOverlap))

10


In [27]:
print(sbertOverlap)

['5080c261-9751-47c0-9cb9-7a9a326fb62f', '560de71d-f0e4-4430-b5b0-e6ceda4b9c20', 'a1ecda1c-7f69-4721-89ae-fb2e6ca9cb2f', '80b6447f-4c37-4194-9862-3785e5075463', '52dbbfda-5b42-4ace-8d59-55cee3eb30c0', '4a4ba70e-d51c-4574-b3f1-42675bef0059', '39c953ff-bb04-4997-a6be-3f4b07e0db56', 'bbc7c651-dd54-46ea-b057-7b001c105912', 'c55aad1c-4eb1-4c9c-a27a-5fbba9196f47', 'dfdf2b81-86fa-48f0-8a47-488ccf3e086b']


3.2. Both Metrics Overlapping S3BERT Unique

In [28]:
s3bertOverlap = list(set(onlyS3bertErrorList) & set(onlyS3bertSentbySentErrorList))

print(len(s3bertOverlap))

7


3.3. Both Metrics Overlapping BERT Unique

In [29]:
bertOverlap = list(set(onlyBertSentbySentErrorList) & set(onlyBertErrorList))

print(len(bertOverlap))

221


In [30]:
print(s3bertOverlap)

['8b3e5896-e1be-4ada-8e49-616acf0ec9b7', '285843c5-b7d9-4ae0-aa25-924a1b3b0dc9', '89cbffbf-4258-47a7-944a-9241ec0c2f17', '6e61a007-5fa7-4275-b45c-9a2d7a8ad2a8', 'dd098674-ee50-4033-a8b9-8637b80b34b6', '88340408-893a-4b3b-8401-c4f4935d2d70', '6b666c82-cfcd-4447-ba48-f91fcb27c170']


3.4. All Three Model Errors

In [31]:
allThreeOverlap = list(set(allThreeSentbySentErrorList) & set(allThreeErrorList))

print(len(allThreeOverlap))

147


In [32]:
print(allThreeOverlap)

['0a17cbf9-397b-4382-9ce0-0fd835626775', '6feb1097-ec20-4267-8428-5db886f7cdc4', '21a8fb15-2612-4fd0-9b10-675eb9473360', '19611623-a65a-4990-8803-559fdf276c3a', '6ab897f0-4f61-40f1-afa6-e9485e44316d', 'cca245f8-ea9f-4110-87c5-9b23b17ce789', '726a0fc9-8c6f-451e-838b-c48140817dfc', 'adaa9d18-572e-40f0-af81-c415abf793a4', '2029aa56-ef8f-4c6f-a599-6dae60566f01', 'dd3506fe-31ce-4bbb-9000-02c222da3466', '1ee70e30-2b21-45be-be71-379c84f910ea', '419d6f93-426a-458b-8e69-0a67e2e06ced', 'f9498236-c0d7-443e-85ea-3ded1a7c270e', '43aa020b-9bca-4af1-9ade-e281f41d04f0', '5af670c5-f7db-4f79-9407-054bb817fe93', '30e8e610-0fbb-4c86-b190-f4934736ab7d', '1a6f72b0-88e5-47b8-92d9-141200921477', 'dba303f0-ede3-4834-aad4-767eadafa744', '847fb1fc-aa6c-443c-98aa-fd4af3847c73', 'f3b25856-827a-42ba-8ca1-0426c1b01219', '23d95ddb-64bd-461e-b5e6-426a8b77c108', '80e87538-52cb-49b0-b0b6-b1caba694be4', '2ab4f4dc-7b16-45f5-8b72-99d8d10be148', '2e62070e-48c9-4aef-8a4c-f304065b9ad5', '72610204-f986-4e0a-ace9-3aff6625dd45',

3.5. BERT & S3BERT Model Errors

In [33]:
bertAnds3bertOverlap = list(set(bertS3bertSentbySentErrorList) & set(bertS3bertErrorList))

print(len(bertAnds3bertOverlap))

4
